# Column Transformations (Data Cleaning using SK Learn)

## Sk Learn functions for Data Cleaning



> Simple Imputer:-
    
    ->   It is used to fill missing values in a column.
    ->   By default, it replaces with mean but we can give it a parameter 'strategy' to change that.

> Ordinal Encoder:-

    ->   It is used to set priority to a column with more than 2 categories like we did earlier in 'class' column.
    ->   We gave highest priority to first class passengers.
    ->   We give a priority list from low priority to high priority.

> One Hot Encoder:-

    ->   Automatcally creates a single column for each category.
    ->   We give 'drop' parameter to drop the first column as well.

> df.iloc[row,column]

    ->   Used to separate 'X' as features and 'Y' as target values.
    ->   Takes row , column indices to separate.

In [54]:
import numpy as np
import pandas as pd

In [55]:
df=pd.read_csv('Titanic.csv')

In [56]:
df=df[['survived','age','pclass','sex','embarked']] #Taking just selected columns

In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   survived  891 non-null    int64  
 1   age       714 non-null    float64
 2   pclass    891 non-null    int64  
 3   sex       891 non-null    object 
 4   embarked  889 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 34.9+ KB


## Test Train Split using SK Learn

In [58]:
from sklearn.model_selection import train_test_split

In [59]:
# In place of 'X' we gave the dataframe without 'survived' column by dropping it and for 'Y' we just gave the 'survived' column which is the target column
X_train,X_test,Y_train,Y_test=train_test_split(df.drop(columns=['survived']), df['survived'], test_size=0.33, random_state=42)

In [60]:
X_train.info()
X_test.info()
Y_train.info()
Y_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 596 entries, 6 to 102
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       478 non-null    float64
 1   pclass    596 non-null    int64  
 2   sex       596 non-null    object 
 3   embarked  595 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 23.3+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 295 entries, 709 to 173
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       236 non-null    float64
 1   pclass    295 non-null    int64  
 2   sex       295 non-null    object 
 3   embarked  294 non-null    object 
dtypes: float64(1), int64(1), object(2)
memory usage: 11.5+ KB
<class 'pandas.core.series.Series'>
Index: 596 entries, 6 to 102
Series name: survived
Non-Null Count  Dtype
--------------  -----
596 non-null    int64
dtypes: int64(1)
memory usage: 9.3 KB
<class 'pandas.

## Filling Missing Values using Simple Imputer by SK Learn

In [61]:
from sklearn.impute import SimpleImputer

In [62]:
si=SimpleImputer(strategy='mean')

Note: SK Learn always expects 2D array input, thats why we use [[]] double square brackets.

In [63]:
# fit_transform function reads the data from the column, remembers the rule from 'si' object as
#calculating mean and transforms all rows with missing values with the mean
X_train_age=si.fit_transform(X_train[['age']])
X_test_age=si.transform(X_test[['age']])

## One Hot and Ordinal Encoding

In [64]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [65]:
oe=OrdinalEncoder(categories=[[1,2,3]]) # Setting priority low to high for pclass values 1, 2, 3

In [66]:
X_train_pclass=oe.fit_transform(X_train[['pclass']])
X_test_pclass=oe.transform(X_test[['pclass']])

In [67]:
# rows like
# 1, 0, 0
# 0, 1, 0
# 0, 0, 1
# is dense matrix, takes a lot of space.
# sparse_output=True converts it to Sparse Matrix which takes less memory and just stores the location of non-zero values:-
# (1,1,1) 1st row ka 1st column is 1
# (2,2,1) 2nd row ka 2nd column is 1
# (3,3,1) 3rd row ka 3rd column is 1

ohe=OneHotEncoder(sparse_output=False,drop='first')

In [68]:
X_train_embarked=ohe.fit_transform(X_train[['embarked']])
X_test_embarked=ohe.transform(X_test[['embarked']])

In [69]:
print(X_train_age.shape)
print(X_train_pclass.shape)
print(X_train_embarked.shape)

X_train_transformed=np.concatenate((X_train_age,X_train_pclass,X_train_embarked),axis=1)
X_test_transformed=np.concatenate((X_test_age,X_test_pclass,X_test_embarked),axis=1)

(596, 1)
(596, 1)
(596, 3)


In [70]:
X_train_transformed

array([[54.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [29.52598326,  2.        ,  1.        ,  0.        ,  0.        ],
       [25.        ,  1.        ,  0.        ,  0.        ,  0.        ],
       ...,
       [41.        ,  2.        ,  0.        ,  1.        ,  0.        ],
       [14.        ,  0.        ,  0.        ,  1.        ,  0.        ],
       [21.        ,  0.        ,  0.        ,  1.        ,  0.        ]])

## Column Transformer

> Rather than separately doing all those column transformations for data cleaning one by one, we can use Column Transformer.

> Column Transformer takes a list of tuples as 'transformers' parameter.

> The tuple should be of structure (name, object_of_transformer, list_of_columns_to_apply_on).

> It also takes a parameter called 'remainder' which can have two values 'passthrough' and 'drop'.

> 'passthrough' means to retain the columns on which no transformation was applied on the dataset.

> 'drop' means to drop the columns on which no transformation was applied. Just keep the ones that were transformed by the Column Transformer.



In [71]:
from sklearn.compose import ColumnTransformer

In [72]:
CT=ColumnTransformer(transformers=[('si',si,['age']), ('oe',oe,['pclass']), ('ohe',ohe,['embarked'])],remainder='passthrough')

In [73]:
X_train_final=CT.fit_transform(X_train)
X_test_final=CT.transform(X_test)

In [74]:
X_train_final

array([[54.0, 0.0, 0.0, 1.0, 0.0, 'male'],
       [29.525983263598327, 2.0, 1.0, 0.0, 0.0, 'male'],
       [25.0, 1.0, 0.0, 0.0, 0.0, 'male'],
       ...,
       [41.0, 2.0, 0.0, 1.0, 0.0, 'male'],
       [14.0, 0.0, 0.0, 1.0, 0.0, 'female'],
       [21.0, 0.0, 0.0, 1.0, 0.0, 'male']], dtype=object)